# AIPerf Use Case 6 — GPU Telemetry with DCGM

Run a deliberately **small synthetic benchmark** while AIPerf collects **GPU metrics** — power, energy, utilization, memory, temperature — natively, so the latency/throughput story comes paired with the resource-side evidence, all inside the standard AIPerf exports.

Based on the official [AIPerf GPU Telemetry tutorial](https://github.com/ai-dynamo/aiperf/blob/main/docs/tutorials/gpu-telemetry.md) — see `docs/reference/aiperf-office-hours.md` for how this relates to the rest of the repo. These use-case notebooks demonstrate AIPerf capabilities directly; the repo's per-scenario bash scripts remain the source of truth for the Model Selection / Sizing suites.

## Table of contents

1. [Overview](#1-overview)
2. [Requirements](#2-requirements)
3. [Telemetry sources and metrics](#3-telemetry-sources-and-metrics)
4. [Test run](#4-test-run)
5. [Results analysis](#5-results-analysis)
6. [FinOps estimation](#6-finops-estimation)

## 1. Overview

Client-side metrics (TTFT, ITL, throughput) tell you *what* the users experience; GPU telemetry tells you *why* and *at what cost*:

- **Saturation evidence** — rising TTFT with GPU utilization pinned near 100% means compute saturation; rising TTFT at 40% utilization points elsewhere (scheduler, KV-cache pressure, network). Without telemetry you are guessing.
- **Cost per token** — power draw during the run gives tokens/sec/W, the number that connects a benchmark to an electricity/infrastructure bill.
- **Sizing headroom** — GPU memory used vs. available shows how much KV-cache room is left at a given concurrency.

AIPerf collects GPU telemetry through two collector backends, selected via the `--gpu-telemetry` flag:

- **DCGM** — scrapes one or more [dcgm-exporter](https://github.com/NVIDIA/dcgm-exporter) HTTP endpoints (Prometheus format). Works when the GPUs are on a *different* machine (the normal case: the exporter runs next to the inference server; this notebook can run anywhere).
- **PyNVML** — queries local NVIDIA GPUs directly via the `nvidia-ml-py` library. Zero infrastructure, but only sees GPUs on the machine running AIPerf.

Two behaviors that are easy to miss:

- The default endpoints `http://localhost:9400/metrics` and `http://localhost:9401/metrics` are **always attempted**, even without the flag — `--gpu-telemetry` controls console display and lets you add custom endpoints.
- Warmup requests are **excluded** from telemetry statistics, same as for latency metrics — one advantage over side-channel polling (e.g. the `dcgmi`/`nvidia-smi` loop in `sizing/scripts/run_content_generation.sh`, which captures the whole wall-clock window including warmup).

## 2. Requirements

- `aiperf` CLI — the setup cell below installs it via pip if it isn't already on `PATH` (repo convention: record the AIPerf version per run).
- A reachable OpenAI-compatible endpoint (NIM, vLLM, TGI, ...) serving the model under test.
- **One telemetry source** (pick the mode in the pre-flight cell):
  - **`dcgm` mode** — a dcgm-exporter reachable from this machine. On the GPU host, the official tutorial starts one with:

    ```bash
    docker run -d --name dcgm-exporter \
      --gpus all --cap-add SYS_ADMIN -p 9401:9400 \
      -e DCGM_EXPORTER_INTERVAL=33 \
      nvcr.io/nvidia/k8s/dcgm-exporter:4.2.0-4.1.0-ubuntu22.04
    ```

    On Kubernetes with the NVIDIA GPU Operator, dcgm-exporter is usually already running as a DaemonSet — point at `http://nvidia-dcgm-exporter.<namespace>:9400/metrics` (see the Notes section for the multi-node caveat).
  - **`pynvml` mode** — NVIDIA GPU(s) on *this* machine; the pre-flight cell installs `nvidia-ml-py` if needed.
- Notebook Python deps: `pip install -r notebooks/requirements.txt` (jupyter, pandas, matplotlib).
- A Hugging Face token (`HF_TOKEN` in the config cell) if the tokenizer repo is gated (e.g. Llama, Mistral).

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "model-selection").exists() else Path.cwd()
assert (REPO_ROOT / "model-selection").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)
print(f"Repo root: {REPO_ROOT}")

if shutil.which("aiperf") is None:
    print("aiperf CLI not found — installing into this kernel's environment ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "aiperf"], check=True)

aiperf_path = shutil.which("aiperf")
assert aiperf_path, "aiperf still not found after install — restart the kernel and rerun this cell."
print(f"aiperf found at: {aiperf_path}")
version = subprocess.run(["aiperf", "--version"], capture_output=True, text=True)
print((version.stdout or version.stderr).strip())


## 3. Telemetry sources and metrics

All forms of the flag (they compose — e.g. `--gpu-telemetry dashboard localhost:9401`):

| Flag form | What it does |
|---|---|
| `--gpu-telemetry` | Scrape the default endpoints (`localhost:9400`/`9401`) and print the telemetry table in the console summary |
| `--gpu-telemetry <url> [<url> ...]` | Add custom dcgm-exporter endpoints (`http://` prefix and `/metrics` suffix optional) — this is how you reach remote GPU nodes |
| `--gpu-telemetry pynvml` | Query local NVIDIA GPUs directly via `nvidia-ml-py` (no DCGM infrastructure) |
| `--gpu-telemetry dashboard` | Live terminal UI panel with real-time GPU metrics (terminal-oriented; renders poorly in Jupyter) |
| `--gpu-telemetry <fields.csv>` | Extend the default metric set with additional DCGM field IDs (memory-copy utilization, SM/memory clocks, ...) |
| `--no-gpu-telemetry` | Disable collection entirely |

The 7 core metrics collected by default (every backend):

| Metric | Unit | Watch it for |
|---|---|---|
| GPU Power Usage | W | Cost; thermal/power throttling risk |
| Energy Consumption | MJ | Total energy of the run (tokens per joule) |
| GPU Utilization | % | Compute saturation |
| GPU Memory Used | GB | KV-cache headroom at this concurrency |
| GPU Temperature | °C | Sustained-load thermal drift |
| XID Errors | count | Driver/hardware faults (should stay 0) |
| Power Violation | count/% | Time throttled below requested clocks |

Which mode to pick: **`dcgm`** is the production-like path (exporter sits next to the GPUs; multi-node capable; same mechanism you'd use from the K8s suites). **`pynvml`** is the zero-setup path when the GPUs are in this machine. Not every metric exists on every GPU model — AIPerf handles missing ones gracefully.

The pre-flight cell below fails fast — *before* the benchmark spends minutes generating load — if the chosen telemetry source isn't actually reachable.

In [ ]:
TELEMETRY_MODE = "dcgm"          # "dcgm" (scrape a dcgm-exporter) or "pynvml" (local GPUs, no DCGM)
DCGM_URLS = ["localhost:9400", "localhost:9401"]   # dcgm-exporter endpoint(s) to probe; unreachable ones are dropped

import urllib.request


def dcgm_metrics_url(url):
    """Normalize the way AIPerf does: http:// prefix and /metrics suffix are optional."""
    if not url.startswith("http"):
        url = "http://" + url
    if not url.rstrip("/").endswith("/metrics"):
        url = url.rstrip("/") + "/metrics"
    return url


if TELEMETRY_MODE == "dcgm":
    reachable = []
    for u in DCGM_URLS:
        full = dcgm_metrics_url(u)
        try:
            with urllib.request.urlopen(full, timeout=5) as resp:
                body = resp.read().decode()
        except OSError as e:
            print(f"skip: {full} not reachable ({e})")
            continue
        dcgm_lines = [l for l in body.splitlines() if l.startswith("DCGM_FI_")]
        assert dcgm_lines, f"{full} responded but exposes no DCGM_FI_* metrics — is this a dcgm-exporter?"
        print(f"OK: {full} — {len(dcgm_lines)} DCGM metric lines, e.g.:")
        for l in dcgm_lines[:3]:
            print(f"    {l}")
        reachable.append(u)
    assert reachable, (
        "No dcgm-exporter reachable at any DCGM_URLS endpoint — start one (see Requirements) "
        "or switch TELEMETRY_MODE to 'pynvml'."
    )
    DCGM_URLS = reachable
elif TELEMETRY_MODE == "pynvml":
    try:
        import pynvml
    except ImportError:
        print("nvidia-ml-py not found — installing into this kernel's environment ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "nvidia-ml-py"], check=True)
        import pynvml
    pynvml.nvmlInit()
    gpu_count = pynvml.nvmlDeviceGetCount()
    assert gpu_count > 0, "pynvml initialized but found no NVIDIA GPUs on this machine."
    for i in range(gpu_count):
        name = pynvml.nvmlDeviceGetName(pynvml.nvmlDeviceGetHandleByIndex(i))
        print(f"GPU {i}: {name.decode() if isinstance(name, bytes) else name}")
    pynvml.nvmlShutdown()
else:
    raise ValueError(f"TELEMETRY_MODE must be 'dcgm' or 'pynvml', got {TELEMETRY_MODE!r}")


## 4. Test run

Set `URL`, then run — the model name is read automatically from the endpoint's `/v1/models` listing, which doubles as a reachability check and saves a copy-paste (set `MODEL` manually only to override, e.g. when the server hosts several models). The workload is intentionally small (the official tutorial's example scale: concurrency 4, 64 requests, ISL 100 / OSL 200) — the point of this use case is the telemetry plumbing, not the load level. To see the GPU metrics move under real pressure, raise `CONCURRENCY`/`REQUEST_COUNT` toward the levels used in the Use Case 1 sweep or the `sizing/` ladder.

In [ ]:
# ---- Required ----------------------------------------------------------
URL = ""         # e.g. "http://localhost:8000" — OpenAI-compatible endpoint base URL
MODEL = ""       # leave empty to auto-detect from {URL}/v1/models; set to override

assert URL, "Set URL in this cell."

# Ask the endpoint what it serves — verifies the URL is reachable and spares
# the user a second copy-paste. Multi-model servers: first model wins.
if not MODEL:
    models_url = URL.rstrip("/") + "/v1/models"
    with urllib.request.urlopen(models_url, timeout=10) as resp:
        served = json.load(resp)["data"]
    assert served, f"{models_url} is reachable but lists no models."
    if len(served) > 1:
        print(f"{models_url} lists {len(served)} models: {[m['id'] for m in served]}")
    MODEL = served[0]["id"]
print(f"URL   : {URL}")
print(f"MODEL : {MODEL}")

# ---- Workload (small on purpose — see section 4) -----------------------
CONCURRENCY = 4
REQUEST_COUNT = 64
ISL = 100                 # input sequence length (tokens)
OSL = 200                 # output sequence length (tokens)
WARMUP_REQUESTS = 1       # excluded from latency *and* telemetry statistics
TOKENIZER = ""            # HF repo id or local dir; empty = use MODEL as tokenizer name
RANDOM_SEED = "42"

# ---- Hugging Face ------------------------------------------------------
# aiperf downloads the tokenizer from Hugging Face. A token is only needed
# for gated repos (e.g. Llama, Mistral) or private tokenizers.
HF_TOKEN = ""
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN  # inherited by the aiperf subprocess

# Relative to REPO_ROOT (aiperf runs with cwd=REPO_ROOT) — keep artifacts under notebooks/.
OUTPUT_DIR = "notebooks/artifacts/aiperf-uc6-gpu-telemetry"


In [ ]:
assert MODEL and URL, "Run the config cell above first — it sets URL and auto-detects MODEL."

# Telemetry args from the pre-flight cell's mode: either the local-driver
# backend, or the list of dcgm-exporter endpoints to scrape.
if TELEMETRY_MODE == "pynvml":
    telemetry_args = ["--gpu-telemetry", "pynvml"]
else:
    telemetry_args = ["--gpu-telemetry", *DCGM_URLS]

# The command, written exactly as you would type it in a terminal.
# .split() turns it into the argument list subprocess expects.
cmd = f"""
aiperf profile
    --model {MODEL}
    --url {URL}
    --endpoint-type chat
    --streaming
    --concurrency {CONCURRENCY}
    --request-count {REQUEST_COUNT}
    --synthetic-input-tokens-mean {ISL}
    --output-tokens-mean {OSL}
    --warmup-request-count {WARMUP_REQUESTS}
    --random-seed {RANDOM_SEED}
    --tokenizer {TOKENIZER or MODEL}
    --artifact-dir {OUTPUT_DIR}
""".split() + telemetry_args

print("$ " + " ".join(cmd))
result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
print(f"\nexit code: {result.returncode}")


## 5. Results analysis

GPU telemetry lands in three places in the artifact directory, alongside the usual exports:

- `gpu_telemetry_export.csv` — the richest form: one row per endpoint × GPU × metric, with full statistics (avg/min/max/p1...p99/std) plus `GPU_Name`/`GPU_UUID`.
- `profile_export_aiperf.csv` — the summary CSV's **third** blank-line-separated section (the one Use Case 1's `read_export()` deliberately ignored).
- `profile_export_aiperf.json` — same data under the `"telemetry_data"` key, next to the run's full configuration.

Because statistics are computed only over the measurement window (warmup excluded), these numbers line up with the latency/throughput statistics from the same run — that's what makes tokens/sec/W below a fair division.

In [ ]:
from io import StringIO

import pandas as pd

artifact_dir = REPO_ROOT / OUTPUT_DIR
telemetry_csv = next(artifact_dir.rglob("gpu_telemetry_export.csv"), None)

if telemetry_csv is not None:
    print(f"Reading {telemetry_csv.relative_to(REPO_ROOT)}")
    telemetry = pd.read_csv(telemetry_csv)
else:
    # Fallback: GPU telemetry is also section 3 of the multi-section summary CSV.
    summary_path = next(artifact_dir.rglob("profile_export_aiperf.csv"), None)
    assert summary_path is not None, f"No exports under {artifact_dir} — did the Test run complete?"
    sections = summary_path.read_text().strip().split("\n\n")
    assert len(sections) >= 3, (
        "No GPU telemetry section in the summary CSV — collection likely failed. "
        "Rerun the pre-flight cell and check the aiperf log for scrape errors."
    )
    print(f"No gpu_telemetry_export.csv — reading section 3 of {summary_path.relative_to(REPO_ROOT)}")
    telemetry = pd.read_csv(StringIO(sections[2]))

pd.set_option("display.max_rows", None)
overview_cols = [c for c in ["Endpoint", "GPU_Index", "GPU_Name", "Metric", "avg", "p50", "p90", "max"]
                 if c in telemetry.columns]
display(telemetry[overview_cols] if overview_cols else telemetry)


In [ ]:
import matplotlib.pyplot as plt


def metric_rows(name_contains):
    return telemetry[telemetry["Metric"].str.lower().str.contains(name_contains)]


# Per-GPU utilization and power — the saturation and cost pictures side by side.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, key, unit) in zip(axes, [("GPU Utilization", "gpu utilization", "%"),
                                         ("GPU Power Usage", "power usage", "W")]):
    rows = metric_rows(key)
    if rows.empty:
        ax.set_title(f"{title} — not reported by this GPU/backend")
        continue
    labels = (rows["GPU_Index"].astype(str) if "GPU_Index" in rows.columns
              else rows.reset_index().index.astype(str))
    ax.bar(labels, rows["avg"], label="avg")
    if "max" in rows.columns:
        ax.plot(range(len(rows)), rows["max"], "k_", markersize=20, label="max")
    ax.set_title(title)
    ax.set_xlabel("GPU index")
    ax.set_ylabel(unit)
    ax.legend()
plt.tight_layout()

# Energy efficiency: system-wide throughput (run-totals section of the summary
# CSV, same extraction as Use Case 1) divided by total average power draw.
summary_path = next(artifact_dir.rglob("profile_export_aiperf.csv"), None)
sections = summary_path.read_text().strip().split("\n\n")
totals = pd.read_csv(StringIO(sections[1])) if len(sections) > 1 else pd.DataFrame(columns=["Metric", "Value"])
tps_rows = totals[totals["Metric"].str.lower().str.contains("output token throughput")]
power_rows = metric_rows("power usage")

if not tps_rows.empty and not power_rows.empty:
    total_tps = float(tps_rows["Value"].iloc[0])
    total_power_w = float(power_rows["avg"].sum())  # summed across GPUs
    print(f"System throughput : {total_tps:,.1f} tokens/sec")
    print(f"Avg power draw    : {total_power_w:,.1f} W across {len(power_rows)} GPU(s)")
    print(f"Energy efficiency : {total_tps / total_power_w:,.2f} tokens/sec per watt "
          f"(= {total_tps / total_power_w:,.2f} tokens per joule)")
else:
    print("Throughput or power rows missing — cannot compute tokens/sec/W for this run.")


## 6. FinOps estimation

The run measured both halves of a cost fraction over the same warmup-excluded window: tokens produced per unit time (run-totals section) and power/energy burned producing them (telemetry). Turning that into money goes in three steps — prices enter only at the last one:

1. **Price-independent efficiency ratios** — tokens per GPU-hour, kWh per million tokens, tokens per joule. These survive any price change and are the numbers worth carrying between customers. One subtlety: DCGM's *Energy Consumption* is a **cumulative counter** (energy since driver load), so the run's energy is `max − min` of the counter — never its `avg`; `avg power × Benchmark Duration` is the cross-check (a large gap between the two means coarse exporter sampling — see the Notes on `DCGM_EXPORTER_INTERVAL`).
2. **Ownership lens** — the same ratios priced two ways. These are *alternatives, not addends*:
   - **Cloud / rented** — an all-in $/GPU-hour rate. Electricity is already inside the rate; adding the energy cost on top would double-count it.
   - **On-prem / owned** — amortized CapEx (purchase price spread over the depreciation horizon) **plus** energy at the local electricity price × PUE. Only here are the two genuinely additive.
3. **Reality adjustments** — capacity cost (the cloud rate, the CapEx) is billed 24/7 whether serving or not, so at fleet utilization *U* every token carries `capacity cost / U`; energy roughly scales with load, so it is not divided (idle draw is a second-order effect). And bill only the GPUs actually serving this model — a dcgm-exporter can report idle GPUs on the same node (`ATTRIBUTED_GPU_INDICES` filters them out).

**Read the output as a method demo, not a price.** At this notebook's deliberately tiny scale the GPU is mostly idle, and per-token cost falls steeply with batching until saturation — so the $ figures below are order-of-magnitude pessimistic. For a real capacity price, run the same math on the `sizing/` ladder's highest healthy rung (every analysis cell here works on any AIPerf artifact dir — just change `OUTPUT_DIR`). Costs count **output tokens only**; input tokens are far cheaper to produce, so compare only against API *output*-token prices.

In [ ]:
# ---- Prices — replace with the customer's actual rates -----------------
ALL_IN_GPU_HOUR_PRICE = 2.00   # $/GPU-hour, cloud/rented all-in rate (replace with your quote)
GPU_PURCHASE_PRICE = 30_000    # $ per GPU (on-prem CapEx)
AMORTIZATION_YEARS = 4         # depreciation horizon for that CapEx
ELECTRICITY_PRICE_KWH = 0.15   # $/kWh (on-prem)
PUE = 1.3                      # datacenter power usage effectiveness (cooling/overhead multiplier)
UTILIZATION = 1.0              # fraction of billed hours actually serving — divides capacity cost only
ATTRIBUTED_GPU_INDICES = None  # e.g. [0, 1] if the exporter also saw GPUs not serving this model
API_PRICE_PER_MTOK = None      # optional: a managed-API $/M *output* tokens, as a sanity anchor


def total_row(name_contains):
    rows = totals[totals["Metric"].str.lower().str.contains(name_contains)]
    return float(rows["Value"].iloc[0]) if not rows.empty else None


def attributed(rows):
    if ATTRIBUTED_GPU_INDICES is None or "GPU_Index" not in rows.columns:
        return rows
    return rows[rows["GPU_Index"].isin(ATTRIBUTED_GPU_INDICES)]


tps = total_row("output token throughput")          # tokens/sec, system-wide
duration_s = total_row("benchmark duration")        # seconds, warmup excluded
req_ps = total_row("request throughput")            # requests/sec
power = attributed(metric_rows("power usage"))      # W, one row per attributed GPU
energy = attributed(metric_rows("energy consumption"))  # cumulative counter, MJ

if tps is None or power.empty:
    print("Throughput or power rows missing — cannot build a FinOps estimate for this run.")
else:
    n_gpus = len(power)
    gpu_label = (", ".join(power["GPU_Index"].astype(str)) if "GPU_Index" in power.columns
                 else f"{n_gpus} (indices not reported)")
    tokens_per_gpu_hour = tps * 3600 / n_gpus

    # Run energy. The Energy Consumption metric is a *cumulative* counter
    # (since driver load), so the run's energy is max - min — never avg.
    counter_kwh = None
    if not energy.empty and {"max", "min"} <= set(energy.columns):
        delta_mj = float((energy["max"] - energy["min"]).sum())
        counter_kwh = delta_mj / 3.6 if delta_mj > 0 else None  # 1 kWh = 3.6 MJ
    derived_kwh = (float(power["avg"].sum()) * duration_s / 3.6e6) if duration_s else None
    run_energy_kwh = counter_kwh if counter_kwh is not None else derived_kwh

    total_out_tok = tps * duration_s if duration_s else None

    print(f"Attributed GPUs        : {gpu_label}")
    print(f"\n-- Efficiency ratios (price-independent) --")
    print(f"Tokens per GPU-hour    : {tokens_per_gpu_hour:,.0f}")
    if counter_kwh is not None and derived_kwh is not None:
        print(f"Run energy             : {counter_kwh:.4f} kWh (counter max-min)  "
              f"vs {derived_kwh:.4f} kWh (avg power x duration) — large gaps mean coarse exporter sampling")
    elif run_energy_kwh is not None:
        print(f"Run energy             : {run_energy_kwh:.4f} kWh")
    kwh_per_mtok = None
    if run_energy_kwh and total_out_tok:
        kwh_per_mtok = run_energy_kwh / total_out_tok * 1e6
        print(f"Energy per 1M tokens   : {kwh_per_mtok:.2f} kWh  "
              f"({total_out_tok / (run_energy_kwh * 3.6e6):.2f} tokens/joule)")

    # Cloud lens: the all-in rate already includes electricity — capacity only.
    cloud_mtok = ALL_IN_GPU_HOUR_PRICE / tokens_per_gpu_hour * 1e6 / UTILIZATION
    print(f"\n-- Cloud / rented ($ {ALL_IN_GPU_HOUR_PRICE:.2f}/GPU-hr all-in, U={UTILIZATION:.0%}) --")
    print(f"$ per 1M output tokens : ${cloud_mtok:,.4f}")

    # On-prem lens: amortized CapEx (capacity, /U) + energy (scales with load).
    capex_hr = GPU_PURCHASE_PRICE / (AMORTIZATION_YEARS * 8760)
    onprem_capex_mtok = capex_hr / tokens_per_gpu_hour * 1e6 / UTILIZATION
    onprem_energy_mtok = kwh_per_mtok * ELECTRICITY_PRICE_KWH * PUE if kwh_per_mtok else 0.0
    onprem_mtok = onprem_capex_mtok + onprem_energy_mtok
    print(f"\n-- On-prem / owned (${GPU_PURCHASE_PRICE:,}/GPU over {AMORTIZATION_YEARS}y = "
          f"${capex_hr:.2f}/GPU-hr, + energy @ ${ELECTRICITY_PRICE_KWH}/kWh x PUE {PUE}) --")
    print(f"$ per 1M output tokens : ${onprem_mtok:,.4f}  "
          f"(CapEx ${onprem_capex_mtok:,.4f} + energy ${onprem_energy_mtok:,.4f})")

    if req_ps:
        tok_per_req = tps / req_ps
        print(f"\n$ per 1K requests      : cloud ${cloud_mtok * tok_per_req / 1000:,.4f}, "
              f"on-prem ${onprem_mtok * tok_per_req / 1000:,.4f}  "
              f"(~{tok_per_req:,.0f} output tokens/request)")
    if API_PRICE_PER_MTOK:
        print(f"\nvs API @ ${API_PRICE_PER_MTOK}/M output tokens: "
              f"cloud lens is {cloud_mtok / API_PRICE_PER_MTOK:,.1f}x the API price at this load")

    conc = globals().get("CONCURRENCY", "?")
    print(f"\nWARNING: measured at concurrency {conc} — a deliberately tiny method demo. Per-token cost")
    print("falls steeply with batching until saturation; for a capacity price, rerun this math on the")
    print("sizing/ ladder's highest healthy rung (point OUTPUT_DIR at that run's artifact dir).")


### Notes

- **Kubernetes**: with the NVIDIA GPU Operator, dcgm-exporter already runs as a DaemonSet on every GPU node — no new infrastructure needed, just `--gpu-telemetry http://nvidia-dcgm-exporter.<namespace>:9400/metrics`. Caveat on multi-GPU-node clusters: the Service round-robins across *all* nodes' exporter pods, so target the exporter pod(s) on the node(s) actually serving the model (pod IPs or a headless service) or the statistics will mix in idle GPUs.
- **Relation to the `sizing/` suite**: `sizing/scripts/run_content_generation.sh` currently captures GPU metrics with a backgrounded `dcgmi dmon`/`nvidia-smi` polling loop into a side-channel `gpu-metrics.csv`. Native `--gpu-telemetry` is the candidate replacement (`docs/reference/aiperf-office-hours.md`): warmup-filtered, aligned to the measurement window, and it lands in the standard AIPerf exports — matching the repo's "output = raw AIPerf export" convention.
- **FinOps at real scale**: the section 6 figures inherit this run's deliberately tiny load. For capacity planning, re-derive $/M tokens at the `sizing/` ladder's highest rung that still meets the latency/goodput targets — that's where batching has pushed tokens-per-GPU-hour to its sustainable maximum, and the per-token cost is the one worth quoting.
- **Finer sampling / more fields**: dcgm-exporter's default collection interval is 30 s — far too coarse for a minutes-long run. The tutorial's `docker run` sets `DCGM_EXPORTER_INTERVAL=33` (ms) for fine-grained profiling; mount a custom fields CSV (`-f /etc/dcgm-exporter/custom.csv`) and pass the same file to `--gpu-telemetry` to collect beyond the 7 core metrics.
- **Healthy baselines**: XID errors and power violations should stay at 0; non-zero power violations mean the GPU throttled below requested clocks during the run — treat throughput measured while throttling as suspect.
- **AMD GPUs**: the same flag also takes `amdsmi` for local AMD Instinct GPUs (ROCm), reporting `amd_`-prefixed equivalents.